In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]

In [3]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [4]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent,
)

In [5]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [6]:
response.output[0].content[0].text

'{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

In [7]:
event = response.output_parsed

In [8]:
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [9]:
response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
    text_format=CalendarEvent
)
print(response.output_text)

{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}


In [10]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Date: Friday  


## Structured RAG

In [11]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 385 chunks from 95 documents


In [12]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [13]:
import json

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [14]:
def llm(user_prompt, instructions=None, model="gpt-4o-mini"):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    return response.output_text

In [15]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)

In [16]:
answer = rag('how do I implement LLM as a Judge?')

In [76]:
def llm_structured(
        user_prompt,
        output_type,
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed



In [18]:
response = llm_structured(
    instructions="Extract the event information.",
    user_prompt="Alice and Bob are going to a science fair on Friday.",
    output_type=CalendarEvent,
)

In [19]:
response

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [20]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool

In [73]:
def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured(
        instructions=instructions,
        user_prompt=prompt,
        output_type=output_type,
    )
 

In [28]:
answer = rag_structured('how do I do llm evals?')

print(answer.answer[:150])
print(answer.found_answer)

To conduct LLM evaluations, you need to follow these steps:

1. **Installation and Imports**:
   - Install the Evidently library using:
     ```python
True


In [29]:
answer = rag_structured('how do I install kafka on windows?')

print(answer.answer[:100])
print(answer.found_answer)

The provided context does not include any information on how to install Kafka on Windows.
False


In [31]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'title': 'Answer', 'type': 'string'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['answer', 'found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [36]:
from typing import Optional

class RAGResponse(BaseModel):
    answer: Optional[str] = None
    found_answer: bool

In [37]:
RAGResponse.model_json_schema()

{'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [39]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [42]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [43]:
instructions = """
You're a documentation assistant.Answer the QUESTION basted on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.

If you don't find the answer, set 'answer' to None
"""

In [44]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [45]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

In [46]:
class RAGResponse(BaseModel):
    """
    The response from the documentation RAG System

    If the answer to the question wasn't found in the database, 'answer' is None.
    """
    answer: Optional[str] = None
    found_answer: bool

In [47]:
RAGResponse.model_json_schema()

{'description': "The response from the documentation RAG System\n\nIf the answer to the question wasn't found in the database, 'answer' is None.",
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Answer'},
  'found_answer': {'title': 'Found Answer', 'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [48]:
answer = rag_structured('how do I install kafka on windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [49]:
from pydantic import Field

class RAGResponse(BaseModel):
    """
    The response from the documentation RAG system
    """
    answer: Optional[str] = Field(None, description="Answer to the question or None if it's not found")
    found_answer: bool = Field(description="True if the answer is found, False otherwise")

In [50]:
RAGResponse.model_json_schema()

{'description': 'The response from the documentation RAG system',
 'properties': {'answer': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'description': "Answer to the question or None if it's not found",
   'title': 'Answer'},
  'found_answer': {'description': 'True if the answer is found, False otherwise',
   'title': 'Found Answer',
   'type': 'boolean'}},
 'required': ['found_answer'],
 'title': 'RAGResponse',
 'type': 'object'}

In [54]:
answer = rag_structured('how do I install kafka on Windows?', RAGResponse)

print(answer.answer)
print(answer.found_answer)

None
False


In [56]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explaination about the condidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comarison", "reference"] = Field(description="The catrgory of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [58]:
RAGResponse.model_json_schema()

{'description': 'This model provides a structured answer with metadata about the about the response,\nincluding confidence, categorization, and follow-up suggestions.',
 'properties': {'answer': {'description': "The main answer to the user's question in markdown",
   'title': 'Answer',
   'type': 'string'},
  'found_answer': {'description': 'True if relevant information was found in the documentation',
   'title': 'Found Answer',
   'type': 'boolean'},
  'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
   'title': 'Confidence',
   'type': 'number'},
  'confidence_explanation': {'description': 'Explaination about the condidence level',
   'title': 'Confidence Explanation',
   'type': 'string'},
  'answer_type': {'description': 'The catrgory of the answer',
   'enum': ['how-to',
    'explanation',
    'troubleshooting',
    'comarison',
    'reference'],
   'title': 'Answer Type',
   'type': 'string'},
  'followup_questions': {'descri

In [60]:
answer = rag_structured('how do I evaluate llms?', RAGResponse)

In [61]:
print(answer.answer[:100])
print(answer.confidence)
print(answer.confidence_explanation)
print(answer.answer_type)
print(answer.followup_questions)

To evaluate LLMs, you can follow these steps:

1. **Set Up Evaluator LLMs**: Pass the API keys for t
0.95
The answer is derived directly from the provided context with clear steps outlined for evaluating LLMs.
how-to
['What tools can I use for LLM evaluation?', 'Can I use self-hosted LLMs for evaluation?', 'What are common criteria for evaluating LLM outputs?']


In [68]:
from pydantic import model_validator


class AnswerNotFound(BaseModel):
    explanation: str


class AnswerResponse(BaseModel):
    """
    If answer is found, 'answer' is populated.
    If no answer is found, 'answer_not_found' is populated.
    Only one of the two fields can be set at a time. Never both or neither.
    """

    answer_not_found: Optional[AnswerNotFound] = None
    found_answer: bool
    answer: Optional[RAGResponse] = None

    @model_validator(mode="after")
    def check_consistency(self):
        if self.answer is not None and self.answer_not_found is not None:
            raise ValueError("Provide either 'answer' or 'answer_not_found', not both.")

        if self.answer is None and self.answer_not_found is None:
            raise ValueError("Provide either 'answer' or 'answer_not_found'.")

        return self

In [69]:
AnswerResponse.model_json_schema()

{'$defs': {'AnswerNotFound': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'}},
   'required': ['explanation'],
   'title': 'AnswerNotFound',
   'type': 'object'},
  'RAGResponse': {'description': 'This model provides a structured answer with metadata about the about the response,\nincluding confidence, categorization, and follow-up suggestions.',
   'properties': {'answer': {'description': "The main answer to the user's question in markdown",
     'title': 'Answer',
     'type': 'string'},
    'found_answer': {'description': 'True if relevant information was found in the documentation',
     'title': 'Found Answer',
     'type': 'boolean'},
    'confidence': {'description': 'Confidence score from 0.0 to 1.0 indicating how certain the answer is',
     'title': 'Confidence',
     'type': 'number'},
    'confidence_explanation': {'description': 'Explaination about the condidence level',
     'title': 'Confidence Explanation',
     'type': 'string'},
    'ans

In [77]:
answer = rag_structured('how do I install kafka on windows?', AnswerResponse)
answer

AnswerResponse(answer_not_found=AnswerNotFound(explanation='The provided context does not contain any information about installing Kafka on Windows.'), found_answer=False, answer=None)

In [75]:
answer = rag_structured('how do I run llm evals?', AnswerResponse)
answer

AnswerResponse(answer_not_found=None, found_answer=True, answer=RAGResponse(answer='To run LLM evaluations, follow these steps:\n\n1. **Connect to Evidently Cloud**: Make sure to connect to Evidently Cloud and create a project.\n   ```python\n   ws = CloudWorkspace(token="YOUR_API_TOKEN", url="https://app.evidently.cloud")\n   project = ws.create_project("Your Project Name", org_id="YOUR_ORG_ID")\n   ```\n\n2. **Prepare Your Dataset**: Create a dataset with your questions and reference answers. For example:\n   ```python\n   data = [\n       ["Question 1", "Reference Answer 1"],\n       ["Question 2", "Reference Answer 2"]\n   ]\n   columns = ["question", "target_response"]\n   ref_data = pd.DataFrame(data, columns=columns)\n   ```\n\n3. **Create a Report**: Use the `TextEvals` preset to run the evaluation.\n   ```python\n   report = Report(metrics=[TextEvals()])\n   my_eval = report.run(ref_dataset, None)\n   ```\n   Where `ref_dataset` is your prepared dataset with descriptors.\n\n4.

In [72]:
from pydantic import ValidationError

try:
    AnswerResponse()
except ValidationError as e:
    print("Validation error:")
    print(e)

Validation error:
1 validation error for AnswerResponse
found_answer
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
